# Step 6 - Build the case bank

Packages everything annotators and LLM judges will see into **one uniform unit per case**, with technique/model identity hidden.

| Item type | Count | What it is |
|---|---|---|
| `case` | 480 | one explanation (profile + recipe + checked facts + explanation text) |
| `diagnostic` | ~15 | a normal-looking case with **one deliberately injected error** (invented preference, wrong nutrition comparison, ignored time limit, ignored dietary requirement) |
| `pair` | 15 | two explanations for the same profile and recipe; the annotator picks the better one |

**Inputs** (Steps 3-5): `profiles_*.jsonl`, `recommendation_pool.jsonl`, `case_inputs.jsonl`, `explanations.jsonl`, `constraint_overrides.csv`, `catalog` (only for the diet diagnostics).

**Outputs** (`processed/case_bank/`):
- `case_bank.jsonl` - what annotators/judges see (no technique, model or role fields)
- `answer_key.csv` - **hidden**: technique, source, role, injected error per `case_id`. Do not share with annotators.
- `pairs.jsonl` / `pair_key.csv`
- `pilot_case_ids.csv`
- `forms/pilot_A1.html ... main_A3.html` - one self-contained annotation form per annotator
- `scoring_sheet_template.csv` - the columns every annotation file must have

Everything is seeded: re-running gives the same IDs. **Re-run this notebook whenever `explanations.jsonl` changes**.

Fixes applied here (found while reviewing Step 5 output: 5_Generating_Explanations.ipynb):
1. grammar glitch left by citation stripping ("Based on your past reviews, tend to like...") - only grounded texts had it, so it would have revealed the technique;
2. profile text artifacts (`... or tablespoons template`, a stray `10 required)`);
3. ingredient lines with lost fractions (`/ cup baking cocoa`) and raw decimals (`0.6666 cup`).

#### 1. Setup

In [1]:
import sys, re, json, random, zlib
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "src" / "config.py").exists():
            return p
    raise FileNotFoundError("Could not find src/config.py above %s. Open the notebook from inside the FoodJudgeBench folder." % here)


ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "src"))
from config import PROC_DIR, as_list, read_table, scrub_text

OUT = PROC_DIR / "case_bank"
FORMS = OUT / "forms"
FORMS.mkdir(parents=True, exist_ok=True)
print("project root:", ROOT)

project root: C:\HP Data\University-Masters\Semester_3\P1_Natural Language Processing\Lab\Project\FoodJudgeBench


#### 2. Settings (the only cell you normally edit)

In [2]:
SEED = 2026
ANNOTATORS = ["A1", "A2", "A3"]      # 3 annotators, as in the research plan
RATE_ALL = True                       # True: everyone rates all 480 cases (~20 h each).
                                      # False: each case gets 2 of the 3 annotators (balanced rotation, ~13 h each).
                                      # Diagnostics, pairs and pilot are always rated by everyone.
N_PILOT = 10                          # pilot items (8 normal cases + 2 diagnostics)
N_PAIRS = 15
DIAG_PLAN = {"invented_preference": 4, "wrong_nutrition": 4, "ignored_time_limit": 4, "ignored_dietary_constraint": 3}
DIAG_BASE_TECHNIQUES = ["template", "direct_llm", "grounded_llm"]   # which writers the injected-error texts are based on
INCLUDE_FRAMED_EXTRAS = False         # goal-framed extras are not part of the 480
TECHNIQUES = ["template", "direct_llm", "grounded_llm", "grounded_check_llm"]

rng = random.Random(SEED)

#### 3. Load inputs and sanity-check them

In [3]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


case_inputs = {c["case_key"]: c for c in read_jsonl(PROC_DIR / "case_inputs.jsonl")}
expl = read_jsonl(PROC_DIR / "explanations.jsonl")
pool = read_jsonl(PROC_DIR / "recommendation_pool.jsonl")
profiles = {p["profile_id"]: p for p in read_jsonl(PROC_DIR / "profiles_historical.jsonl") + read_jsonl(PROC_DIR / "profiles_synthetic.jsonl")}
ov_path = PROC_DIR / "constraint_overrides.csv"
overrides = pd.read_csv(ov_path) if ov_path.exists() else pd.DataFrame(columns=["recipe_id", "constraint", "true_status"])

neutral = [e for e in expl if e.get("framing", "neutral") == "neutral" and e.get("text")]
by_case = defaultdict(dict)
for e in neutral:
    by_case[e["case_key"]][e["technique"]] = e

problems = []
for ck, c in case_inputs.items():
    missing = [t for t in TECHNIQUES if t not in by_case.get(ck, {})]
    if missing:
        problems.append(f"{ck}: missing {missing}")
print(f"{len(case_inputs)} case inputs, {len(neutral)} neutral explanations, {len(profiles)} profiles")
assert not problems, "Some cases lack a technique:\n" + "\n".join(problems[:10])
assert len(case_inputs) * len(TECHNIQUES) == len(neutral), "expected exactly 4 neutral explanations per case"
print("OK: every case has all four techniques")

120 case inputs, 480 neutral explanations, 60 profiles
OK: every case has all four techniques


#### 4. Cleaning helpers (text repair, profile card, recipe card, evidence)

In [4]:
DEC = {0.125: "1/8", 0.25: "1/4", 0.3333: "1/3", 0.375: "3/8", 0.5: "1/2", 0.6667: "2/3", 0.75: "3/4"}
n_fixed_lines = Counter()


def clean_ingredient_line(s: str) -> str:
    s = scrub_text(str(s)).strip()
    s = re.sub(r"\$template\d*\$", "(another ingredient)", s)

    def dec(m):
        v = round(float(m.group(0)), 4)
        for k, lab in DEC.items():
            if abs(v - k) < 0.002:
                n_fixed_lines["decimal"] += 1
                return lab
        return f"{float(m.group(0)):.2f}".rstrip("0").rstrip(".")
    s = re.sub(r"\b\d+\.\d{3,}\b", dec, s)
    if s.startswith("/ ") or s.startswith("/"):      # numerator was lost upstream
        n_fixed_lines["lost_fraction"] += 1
        s = "some " + s.lstrip("/ ").strip()
    return s


def repair_text(t: str) -> str:
    """Repairs artifacts left by citation stripping so technique cannot be guessed from them."""
    t = re.sub(r"\s+", " ", scrub_text(t)).strip()
    t = re.sub(r"^Based on your past reviews, (tend|like|prefer|enjoy|have)\b", r"Based on your past reviews, you \1", t)
    t = re.sub(r"\s+(in|within|across|from) the dataset\b", "", t)          # "in the dataset" appears only in grounded texts
    t = re.sub(r"\s*[\[(]?MUST MENTION[^\])]*[\])]?", "", t)
    t = re.sub(r",?\s*(?:as )?(?:\w+ )?in facts? F\d+(?:\s*(?:,|and|&)\s*F\d+)*", "", t)
    t = re.sub(r"\s+([,.;:])", r"\1", t)
    return t


def clean_profile_text(t: str) -> str:
    t = scrub_text(t)
    t = re.sub(r",?\s*[^,.:;]*\btemplate\d*\b[^,.:;]*(?=[,.;])", "", t)          # e.g. "butter or tablespoons template"
    t = re.sub(r"dislikes \(only \d+ low-rated recipes\); 10 required\)", "dislikes (too few low-rated recipes to infer any)", t)
    t = re.sub(r"dislikes \(only \d+ low-rated recipes\)", "dislikes (too few low-rated recipes to infer any)", t)
    t = t.replace("_", " ")
    return re.sub(r"\s+", " ", t).strip()


def profile_card(profile_text: str) -> list[str]:
    return [s.strip() for s in re.split(r"(?<=[.])\s+(?=[A-Z])", clean_profile_text(profile_text)) if s.strip()]


def evidence_lines(facts) -> list[str]:
    out = []
    for f in facts:
        if f["kind"] == "profile":
            continue
        if f["text"].startswith("The recipe is called") or f["text"].startswith("Ingredients (as listed)"):
            continue
        out.append(re.sub(r"\s+(in|within|across|from) the dataset\b", "", re.sub(r"\s+", " ", scrub_text(f["text"]))))
    return out


def recipe_block(c) -> dict:
    return {"title": scrub_text(c["title"]), "ingredients": [clean_ingredient_line(x) for x in c["ingredient_lines"]]}


# quick look at what the repair touches
before = [e["text"] for e in neutral]
after = [repair_text(t) for t in before]
print("texts changed by repair_text:", sum(a != b for a, b in zip(after, before)))
print("profile texts changed:", sum(clean_profile_text(c["profile_text"]) != c["profile_text"] for c in {c["profile_id"]: c for c in case_inputs.values()}.values()))
bad = [p for p in {c["profile_id"]: c for c in case_inputs.values()}.values() if "template" in clean_profile_text(p["profile_text"]).lower()]
assert not bad, "profile text still contains 'template'"

texts changed by repair_text: 98
profile texts changed: 39


#### 5. Normal cases (480)

In [5]:
items = []          # annotator-facing records
keys = []           # hidden answer key rows


def base_record(ck: str) -> dict:
    c = case_inputs[ck]
    return {
        "profile_id": c["profile_id"],
        "profile_card": profile_card(c["profile_text"]),
        "recipe": recipe_block(c),
        "evidence": evidence_lines(c["facts"]),
        "has_goal": c.get("goal_key") is not None,
        "has_constraint": bool(c["profile_view"]["constraints"]),
        "has_time_limit": c["profile_view"]["time"]["value"] is not None,
    }


for ck, c in case_inputs.items():
    rec = base_record(ck)
    t3, t4 = by_case[ck]["grounded_llm"]["text"], by_case[ck]["grounded_check_llm"]["text"]
    for tech in TECHNIQUES:
        e = by_case[ck][tech]
        r = dict(rec)
        r.update({"item_type": "case", "explanation": repair_text(e["text"]), "_ck": ck})
        items.append(r)
        keys.append({"case_key": ck, "item_type": "case", "technique": tech, "profile_source": c["profile_source"], "role": c["role"],
                     "model": e.get("model"), "injected_error": "", "base_technique": "", "label_uncertain": c.get("label_uncertain", False),
                     "t3_equals_t4": t3 == t4 if tech in ("grounded_llm", "grounded_check_llm") else ""})
print(len(items), "normal cases built")
print("ingredient-line repairs:", dict(n_fixed_lines))

480 normal cases built
ingredient-line repairs: {'lost_fraction': 47, 'decimal': 61}


#### 6. Diagnostic cases (one injected error each)

Every diagnostic is a normal-looking case whose explanation contains **one** known error. The answer key records which. Notes:
- the ordinary explanations may also contain *natural* errors (technique 2 invents some); diagnostics are the only cases where the error is known for certain;
- the **dietary** diagnostics use the pool's `violates_constraint` candidates, minus any candidate the hand-check (`constraint_overrides.csv`) says is not a certain violation.

In [6]:
INVENTED = [
    ("household", "You also told us you are cooking for a large family, so the portions will go a long way."),
    ("spicy", "You have said you prefer spicy food, and this recipe suits that."),
    ("marathon", "As someone who trains for marathons, you will appreciate the extra energy it provides."),
    ("weekend", "You mentioned that you enjoy long weekend cooking projects, which makes this a good fit."),
    ("kids", "Since your children are picky eaters, this is a safe choice for the whole table."),
    ("italian", "You have told us that Italian cuisine is your favourite, and this fits right in."),
]
NUTR_WORD = {"calories": "calories", "sodium": "sodium", "sugar": "sugar", "saturated fat": "saturated fat", "protein": "protein", "fiber": "fiber"}


def sentences(t):
    return re.split(r"(?<=[.!?])\s+", t.strip())


def add_sentence(text, sent, where):
    s = sentences(text)
    if where == "after_first" and len(s) > 1:
        return " ".join([s[0], sent] + s[1:])
    return text.rstrip() + " " + sent


def new_item(rec, text, kind, key_extra):
    r = dict(rec); r.update({"item_type": "diagnostic", "explanation": text})
    items.append(r)
    keys.append({"item_type": "diagnostic", "profile_source": key_extra.pop("profile_source"), "role": key_extra.pop("role", ""),
                 "injected_error": kind, "model": "", "technique": "diagnostic", "label_uncertain": False, "t3_equals_t4": "", **key_extra})


used_profiles = set()
candidates = [ck for ck, c in case_inputs.items()]
rng.shuffle(candidates)


def pick_cases(pred, n):
    out = []
    for ck in candidates:
        c = case_inputs[ck]
        if c["profile_id"] in used_profiles or not pred(c):
            continue
        out.append(ck); used_profiles.add(c["profile_id"])
        if len(out) == n:
            break
    return out


def base_text(ck):
    tech = rng.choice(DIAG_BASE_TECHNIQUES)
    return tech, repair_text(by_case[ck][tech]["text"])


# 6a invented preference
for ck in pick_cases(lambda c: True, DIAG_PLAN["invented_preference"]):
    c = case_inputs[ck]; ptxt = c["profile_text"].lower()
    options = [s for kw, s in INVENTED if kw not in ptxt]
    sent = rng.choice(options)
    tech, txt = base_text(ck)
    new_item(base_record(ck), add_sentence(txt, sent, rng.choice(["end", "after_first"])), "invented_preference",
             {"case_key": ck, "base_technique": tech, "profile_source": c["profile_source"], "role": c["role"], "detail": sent})

# 6b wrong nutrition comparison (claims the opposite of the checked percentile)
def extreme_metric(c):
    pc = c["pool_facts"]["nutrition_percentiles"]
    cand = [(abs(v - .5), k, v) for k, v in pc.items() if k in NUTR_WORD and v is not None and abs(v - .5) >= .25]
    return max(cand) if cand else None


for ck in pick_cases(lambda c: extreme_metric(c) is not None, DIAG_PLAN["wrong_nutrition"]):
    c = case_inputs[ck]; _, m, v = extreme_metric(c)
    sent = (f"It is also among the lowest in {NUTR_WORD[m]} of all the recipes you could choose." if v >= .75
            else f"It is also among the highest in {NUTR_WORD[m]} of all the recipes you could choose.")
    tech, txt = base_text(ck)
    new_item(base_record(ck), add_sentence(txt, sent, "end"), "wrong_nutrition",
             {"case_key": ck, "base_technique": tech, "profile_source": c["profile_source"], "role": c["role"], "detail": f"{m} percentile {v:.2f}: {sent}"})

# 6c ignored time limit: a recipe that really takes longer than the stated limit, described as fitting
def compare_sentence(c):
    for f in c["facts"]:
        if f["text"].startswith("Compared with other recipes"):
            return re.sub(r"^Compared with other recipes in the dataset, this recipe is", "Compared with other recipes, it is", f["text"])
    return ""


time_profiles = [c for c in case_inputs.values() if c["profile_view"]["time"]["value"] is not None and not c["profile_view"]["constraints"]
                 and c["profile_source"] == "synthetic"]
rng.shuffle(time_profiles)
donors = [c for c in case_inputs.values()]
made = 0
for pc_ in time_profiles:
    if made >= DIAG_PLAN["ignored_time_limit"] or pc_["profile_id"] in used_profiles:
        continue
    L = pc_["profile_view"]["time"]["value"]
    dislikes = [(d["item"] if isinstance(d, dict) else str(d)).lower() for d in pc_["profile_view"].get("dislikes", [])]
    ok = [d for d in donors if d["pool_facts"]["time"]["duration_minutes"] and d["pool_facts"]["time"]["duration_minutes"] >= L + 20
          and not any(x in " ".join(d["ingredient_lines"]).lower() for x in dislikes)]
    if not ok:
        continue
    d = rng.choice(ok); dur = int(d["pool_facts"]["time"]["duration_minutes"])
    rec = base_record(pc_["case_key"])
    ev = [e for e in evidence_lines(d["facts"]) if not e.startswith("Time check") and not e.startswith("The recipe features liked") and "It takes about" not in e]
    ev.insert(0, f"It takes about {dur} minutes in total.")
    ev.append(f"Time check: {dur} minutes is over the {L}-minute limit.")
    rec.update({"recipe": recipe_block(d), "evidence": ev})
    txt = f'I recommend "{rec["recipe"]["title"]}" for you. It fits comfortably within your {L}-minute time limit and makes an easy meal. {compare_sentence(d)}'.strip()
    new_item(rec, txt, "ignored_time_limit", {"case_key": pc_["case_key"], "donor_recipe_case": d["case_key"], "base_technique": "injected_template",
                                              "profile_source": "synthetic", "role": "", "detail": f"limit {L} min, recipe {dur} min, text says it fits"})
    used_profiles.add(pc_["profile_id"]); made += 1
print("time-limit diagnostics:", made)

# 6d ignored dietary requirement (recipe that really violates it, described as fitting)
viol = [p for p in pool if p["role"] == "violates_constraint"]
ov = {(int(r.recipe_id), r.constraint): r.true_status for r in overrides.itertuples()}
sure = []
for p in viol:
    chk = p["facts"]["constraint_checks"][0]
    status = ov.get((int(p["recipe_id"]), chk["constraint"]))      # None = never flagged by the hand-check
    if status in (None, "violates"):
        sure.append(p)
print(f"violating candidates: {len(viol)}; certain after hand-check overrides: {len(sure)}")
ids = [int(p["recipe_id"]) for p in sure]
try:
    cat = read_table("catalog", usecols=["new_recipe_id", "title", "ingredients_display"], filter_col="new_recipe_id", filter_values=ids)
    cat = cat.set_index("new_recipe_id")
except Exception as ex:                      # catalog missing: skip these diagnostics, tell the user
    print("WARNING: could not read the catalog; dietary diagnostics skipped:", ex); cat = pd.DataFrame()
rng.shuffle(sure)
made = 0
for p in sure:
    if made >= DIAG_PLAN["ignored_dietary_constraint"] or p["profile_id"] in used_profiles or int(p["recipe_id"]) not in getattr(cat, "index", []):
        continue
    row = cat.loc[int(p["recipe_id"])]; f = p["facts"]; chk = f["constraint_checks"][0]
    pv_case = next((c for c in case_inputs.values() if c["profile_id"] == p["profile_id"]), None)
    if pv_case is None:
        continue
    rec = base_record(pv_case["case_key"])
    lines = [clean_ingredient_line(x) for x in as_list(row["ingredients_display"])]
    n = f["nutrition_per_serving"]
    ev = [f"It takes about {int(f['time']['duration_minutes'])} minutes in total." if f["time"]["duration_minutes"] else "Total time is not listed.",
          f"Per serving: {n['calories']:.0f} kcal, {n['protein_g']:.0f} g protein, {n['total_fat_g']:.0f} g fat ({n['saturated_fat_g']:.0f} g saturated), "
          f"{n['carbs_g']:.0f} g carbohydrate ({n['sugars_g']:.0f} g sugar, {n['fiber_g']:.0f} g fibre), {n['sodium_mg']:.0f} mg sodium.",
          f"Dietary check ({chk['constraint']}): does not pass; this was checked by hand."]
    liked = f.get("matches_liked") or []
    if liked:
        ev.append("The recipe features liked items: " + ", ".join(liked) + ".")
    rec.update({"recipe": {"title": scrub_text(row["title"]), "ingredients": lines}, "evidence": ev})
    txt = (f'"{rec["recipe"]["title"]}" is a good match for you: it fits your {chk["constraint"]} requirement. '
           + (f"It also features {', '.join(liked)}, which you like." if liked else "It is a simple, satisfying choice."))
    new_item(rec, txt, "ignored_dietary_constraint", {"case_key": pv_case["case_key"], "recipe_id": int(p["recipe_id"]), "base_technique": "injected_template",
                                                      "profile_source": p["profile_source"], "role": "violates_constraint",
                                                      "detail": f"{chk['constraint']} violated; text says it fits"})
    used_profiles.add(p["profile_id"]); made += 1
print("dietary diagnostics:", made)
dk = pd.DataFrame([k for k in keys if k["item_type"] == "diagnostic"])
print(dk["injected_error"].value_counts().to_string())

time-limit diagnostics: 4
violating candidates: 15; certain after hand-check overrides: 12
dietary diagnostics: 3
injected_error
invented_preference           4
wrong_nutrition               4
ignored_time_limit            4
ignored_dietary_constraint    3


**Check the diagnostics by eye before annotation** (the dietary ones especially: confirm each recipe really violates the stated diet).

In [7]:
for it, k in zip(items, keys):
    if it["item_type"] == "diagnostic":
        print(f"[{k['injected_error']}] profile {it['profile_id']} | {it['recipe']['title']}\n   {k['detail']}\n   TEXT: {it['explanation'][:260]}\n")

[invented_preference] profile H008 | Caramelized Onions
   You mentioned that you enjoy long weekend cooking projects, which makes this a good fit.
   TEXT: Caramelized Onions is one option we can offer you. You mentioned that you enjoy long weekend cooking projects, which makes this a good fit. It does not clearly match the ingredients and themes you usually favour. It takes about 55 minutes. It differs from the 

[invented_preference] profile S006 | Sesame Pea Pods
   You have told us that Italian cuisine is your favourite, and this fits right in.
   TEXT: I think I have just the thing for you! You have told us that Italian cuisine is your favourite, and this fits right in. This Sesame Pea Pods recipe is a great fit for you because it's gluten-free, which aligns with your dietary needs. The ingredients are also 

[invented_preference] profile S001 | Go-Go Garlic Bread
   Since your children are picky eaters, this is a safe choice for the whole table.
   TEXT: I think you'll really en

#### 7. Pairwise comparison items
Two explanations for the same profile and recipe; the annotator chooses which is better. A/B position is randomised; the key records which technique was shown as A.

In [8]:
PAIR_TYPES = [("direct_llm", "grounded_llm", 4), ("template", "grounded_llm", 4), ("direct_llm", "template", 4), ("grounded_llm", "grounded_check_llm", 3)]
assert sum(n for *_, n in PAIR_TYPES) == N_PAIRS
pairs, pair_keys = [], []
used_pair_cases = set()
ck_list = list(case_inputs)
for t1, t2, n in PAIR_TYPES:
    rng.shuffle(ck_list)
    got = 0
    for ck in ck_list:
        if ck in used_pair_cases:
            continue
        a, b = repair_text(by_case[ck][t1]["text"]), repair_text(by_case[ck][t2]["text"])
        if a == b:
            continue
        swap = rng.random() < .5
        first, second = (t2, t1) if swap else (t1, t2)
        txts = {t1: a, t2: b}
        rec = base_record(ck)
        rec.update({"item_type": "pair", "explanation_A": txts[first], "explanation_B": txts[second]})
        pairs.append(rec)
        pair_keys.append({"case_key": ck, "item_type": "pair", "A": first, "B": second, "profile_source": case_inputs[ck]["profile_source"]})
        used_pair_cases.add(ck); got += 1
        if got == n:
            break
    assert got == n, f"could not find {n} pairs for {t1} vs {t2}"
print(len(pairs), "pairs;", Counter((k["A"], k["B"]) for k in pair_keys).most_common())

15 pairs; [(('grounded_llm', 'template'), 4), (('direct_llm', 'template'), 3), (('direct_llm', 'grounded_llm'), 2), (('grounded_llm', 'direct_llm'), 2), (('grounded_check_llm', 'grounded_llm'), 2), (('template', 'direct_llm'), 1), (('grounded_llm', 'grounded_check_llm'), 1)]


#### 8. Assign IDs (identity hidden) and write the answer key
IDs are random-looking (`C-7F3A2`), carry no information about technique, profile or order, and are stable for a given seed.

In [9]:
def make_ids(n, prefix):
    r = random.Random(SEED + zlib.crc32(prefix.encode()) % 1000)
    pool_ = r.sample(range(0x10000, 0xFFFFF), n)
    return [f"{prefix}-{x:05X}" for x in pool_]


for it, k, cid in zip(items, keys, make_ids(len(items), "C")):
    it["case_id"] = cid; k["case_id"] = cid
for it, k, pid in zip(pairs, pair_keys, make_ids(len(pairs), "P")):
    it["case_id"] = pid; k["case_id"] = pid
assert len({i["case_id"] for i in items + pairs}) == len(items) + len(pairs)

# the annotator-facing fields only (explicit whitelist: nothing else can leak)
PUBLIC = ["case_id", "item_type", "profile_card", "recipe", "evidence", "explanation", "explanation_A", "explanation_B"] 
public_items = [{k: i[k] for k in PUBLIC if k in i} for i in items + pairs]

with open(OUT / "case_bank.jsonl", "w", encoding="utf-8") as f:
    for i in public_items:
        f.write(json.dumps(i, ensure_ascii=False) + "\n")
key_df = pd.DataFrame(keys + pair_keys)
key_cols = ["case_id", "item_type", "case_key", "technique", "injected_error", "base_technique", "profile_source", "role", "model", "t3_equals_t4",
            "label_uncertain", "A", "B", "detail", "donor_recipe_case", "recipe_id"]
key_df = key_df.reindex(columns=key_cols)
key_df.to_csv(OUT / "answer_key.csv", index=False)
print("wrote case_bank.jsonl:", len(public_items), "items;  answer_key.csv: HIDDEN - do not share with annotators")

wrote case_bank.jsonl: 510 items;  answer_key.csv: HIDDEN - do not share with annotators


#### 9. Leak check: nothing visible may reveal technique, model or ids

In [10]:
LEAK = re.compile(r"\b(template|direct_llm|grounded|llama|qwen|gemma|mistral|gpt|claude|ollama|F\d{1,2}\b|\[F\d)", re.I)
leaks = []
for i in public_items:
    blob = json.dumps({k: v for k, v in i.items() if k != "case_id"}, ensure_ascii=False)
    for m in LEAK.finditer(blob):
        leaks.append((i["case_id"], m.group(0)))
print("possible leaks:", Counter(x for _, x in leaks).most_common(10))
# 'template' can legitimately appear in recipe text only as the cleaned placeholder; anything else is a real leak
assert not [l for l in leaks if l[1].lower() not in ("template",)], "identity leak in visible fields"
print("OK: no technique/model identifiers in annotator-facing fields")

# style fingerprints that remain (informational; cannot be removed without rewriting the texts)
fp = pd.DataFrame([{"technique": k["technique"], "first_person": bool(re.search(r"\bI('m|'ve| think| recommend| picked)?\b", i["explanation"])),
                    "mentions_dataset": "dataset" in i["explanation"], "words": len(i["explanation"].split())}
                   for i, k in zip(items, keys) if k["item_type"] == "case"])
print(fp.groupby("technique").agg(first_person=("first_person", "mean"), mentions_dataset=("mentions_dataset", "mean"), mean_words=("words", "mean")).round(2))

possible leaks: []
OK: no technique/model identifiers in annotator-facing fields
                    first_person  mentions_dataset  mean_words
technique                                                     
direct_llm                  0.72               0.0       82.83
grounded_check_llm          0.16               0.0       78.33
grounded_llm                0.16               0.0       81.38
template                    0.00               0.0       56.67


#### 10. Assignment, order and pilot set
Each annotator gets their own random order (so position/fatigue effects are not shared), with diagnostics and pairs mixed in.

In [11]:
main_ids = [k["case_id"] for k in keys if k["item_type"] == "case"]
diag_ids = [k["case_id"] for k in keys if k["item_type"] == "diagnostic"]
pair_ids = [k["case_id"] for k in pair_keys]

assign = {a: [] for a in ANNOTATORS}
order_rng = random.Random(SEED + 1)
shuffled = main_ids[:]; order_rng.shuffle(shuffled)
for n, cid in enumerate(shuffled):
    if RATE_ALL:
        who = ANNOTATORS
    else:
        who = [ANNOTATORS[n % len(ANNOTATORS)], ANNOTATORS[(n + 1) % len(ANNOTATORS)]]
    for a in who:
        assign[a].append(cid)
for a in ANNOTATORS:
    ids_a = assign[a] + diag_ids + pair_ids
    random.Random(SEED + 10 + ANNOTATORS.index(a)).shuffle(ids_a)
    assign[a] = ids_a
print({a: len(v) for a, v in assign.items()}, "items per annotator (cases + diagnostics + pairs)")
per_case = Counter(c for a in ANNOTATORS for c in assign[a] if c in set(main_ids))
print("annotators per normal case:", Counter(per_case.values()))

# pilot: 2 normal cases per technique from different profiles + 2 diagnostics of different types
pil_rng = random.Random(SEED + 2)
pilot, seen_prof = [], set()
kd = key_df.set_index("case_id")
for tech in TECHNIQUES:
    cand = [k for k in keys if k["item_type"] == "case" and k["technique"] == tech]
    pil_rng.shuffle(cand)
    n_ = 0
    for k in cand:
        pid = case_inputs[k["case_key"]]["profile_id"]
        if pid in seen_prof:
            continue
        seen_prof.add(pid); pilot.append(k["case_id"]); n_ += 1
        if n_ == (N_PILOT - 2) // len(TECHNIQUES):
            break
pilot += [k["case_id"] for k in keys if k["item_type"] == "diagnostic"][: N_PILOT - len(pilot)]
pd.DataFrame({"case_id": pilot}).to_csv(OUT / "pilot_case_ids.csv", index=False)
print(len(pilot), "pilot items (they are re-rated in the main round; pilot ratings are discarded after the wording is fixed)")

{'A1': 510, 'A2': 510, 'A3': 510} items per annotator (cases + diagnostics + pairs)
annotators per normal case: Counter({3: 480})
10 pilot items (they are re-rated in the main round; pilot ratings are discarded after the wording is fixed)


#### 11. The annotation form (one self-contained HTML file per annotator)
The form shows one case at a time, enforces the rubric (including "not applicable" options), autosaves in the browser, and exports a CSV. The questions are defined once in `RUBRIC` below - the same list is exported as `scoring_sheet_template.csv` and reused by the LLM-judge step, so humans and judges answer identical questions.

In [12]:
RUBRIC = [
    # id, section, label, question, options (value, text), applies
    {"id": "ignored_constraint", "section": "A. Problems (answer for every case)", "type": "yn_na",
     "label": "Ignores a stated requirement",
     "q": "Does the explanation ignore or contradict something the profile STATES as a requirement (a diet or allergy, or a time limit)? Wording like \"fits your vegan diet\" for a recipe that is not vegan counts as YES.",
     "na": "Profile states no diet/allergy/time requirement"},
    {"id": "factual_error", "section": "A. Problems (answer for every case)", "type": "yn",
     "label": "Wrong fact about the recipe",
     "q": "Does it say anything about the RECIPE that is wrong - ingredients, time, nutrition numbers, or how it compares with other recipes? Check against the recipe and the checked facts."},
    {"id": "invented_personalization", "section": "A. Problems (answer for every case)", "type": "yn",
     "label": "Invents something about the person",
     "q": "Does it claim something about the PERSON that the profile does not say (a preference, habit, family situation, goal)?"},
    {"id": "unsupported_claim", "section": "A. Problems (answer for every case)", "type": "yn",
     "label": "Other unsupported claim",
     "q": "Apart from the two problems above, does it state something as fact that you cannot check from what you were given (for example health benefits, or taste described as certain fact)?"},
    {"id": "overstated_certainty", "section": "A. Problems (answer for every case)", "type": "yn_na",
     "label": "Treats a guess as certain",
     "q": "The profile says whether information was STATED by the person or only INFERRED from past behaviour. Does the explanation present something inferred as if it were confirmed (\"you love...\", \"you always...\")?",
     "na": "Profile is fully stated (nothing inferred)"},
    {"id": "clarity", "section": "B. Quality (1 = very poor, 5 = excellent)", "type": "scale",
     "label": "Clarity", "q": "Is it easy to read and understand - clear, well organised, not confusing or repetitive?"},
    {"id": "profile_fit", "section": "B. Quality (1 = very poor, 5 = excellent)", "type": "scale",
     "label": "Fit with the profile", "q": "Does it use the profile information that matters for THIS person, correctly and proportionately? (Judge suitability for the stated profile, not whether YOU like the dish.)"},
    {"id": "goal_alignment", "section": "B. Quality (1 = very poor, 5 = excellent)", "type": "scale_na",
     "label": "Goal alignment", "q": "If the profile has a health goal: does the explanation address it correctly, including saying so honestly when the recipe does NOT help the goal?",
     "na": "Profile states no health goal"},
    {"id": "tradeoff", "section": "B. Quality (1 = very poor, 5 = excellent)", "type": "scale_na",
     "label": "Trade-offs", "q": "Does it honestly tell the person about downsides or mismatches (for example high sodium, longer than usual, an unmet goal)? 5 = clear and honest; 1 = hides or ignores real downsides.",
     "na": "This recipe has no downside for this person"},
    {"id": "overall", "section": "B. Quality (1 = very poor, 5 = excellent)", "type": "scale",
     "label": "Overall suitability", "q": "Overall, how suitable and trustworthy is this explanation for THIS person and THIS recipe?"},
]
PAIR_REASONS = ["more accurate", "clearer", "fits the profile better", "communicates trade-offs better", "other"]
template_cols = ["annotator_id", "mode", "case_id", "item_type"] + [r["id"] for r in RUBRIC] + ["pair_choice", "pair_reason", "comment", "seconds", "saved_at"]
pd.DataFrame(columns=template_cols).to_csv(OUT / "scoring_sheet_template.csv", index=False)
json.dump({"rubric": RUBRIC, "pair_reasons": PAIR_REASONS, "columns": template_cols}, open(OUT / "rubric.json", "w", encoding="utf-8"), indent=1)
print("rubric questions:", [r["id"] for r in RUBRIC])

rubric questions: ['ignored_constraint', 'factual_error', 'invented_personalization', 'unsupported_claim', 'overstated_certainty', 'clarity', 'profile_fit', 'goal_alignment', 'tradeoff', 'overall']


In [13]:
FORM_HTML = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Food explanation rating</title>
<style>
:root{--bg:#f6f5f2;--card:#fff;--ink:#1d1d1b;--mute:#6b6a66;--line:#dcdad3;--acc:#1f6f5c;--accbg:#e4f1ed;--warn:#b3261e;--hl:#fff8e1}
@media (prefers-color-scheme:dark){:root{--bg:#161615;--card:#1f1f1d;--ink:#ececE8;--mute:#9a9893;--line:#363633;--acc:#5cc4a8;--accbg:#1e3a33;--warn:#f2a29b;--hl:#2c2819}}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:16px/1.5 system-ui,sans-serif}
header{position:sticky;top:0;background:var(--card);border-bottom:1px solid var(--line);padding:10px 16px;display:flex;gap:12px;align-items:center;flex-wrap:wrap;z-index:5}
header b{font-size:15px}.bar{flex:1;min-width:120px;height:8px;background:var(--line);border-radius:4px;overflow:hidden}.bar i{display:block;height:100%;background:var(--acc)}
button{font:inherit;border:1px solid var(--line);background:var(--card);color:var(--ink);padding:8px 14px;border-radius:8px;cursor:pointer}button.p{background:var(--acc);color:#fff;border-color:var(--acc)}button:disabled{opacity:.45;cursor:default}
main{max-width:1100px;margin:0 auto;padding:16px;display:grid;gap:16px;grid-template-columns:1fr}
@media(min-width:960px){main{grid-template-columns:minmax(0,1.05fr) minmax(0,1fr)}}
.card{background:var(--card);border:1px solid var(--line);border-radius:12px;padding:14px 16px}.card h3{margin:0 0 8px;font-size:13px;letter-spacing:.04em;text-transform:uppercase;color:var(--mute)}
.card ul{margin:0;padding-left:18px}.card li{margin:3px 0}
.expl{background:var(--hl);border-color:#e6d48a;font-size:17px}.expl p{margin:0}
.q{padding:12px 0;border-top:1px solid var(--line)}.q:first-of-type{border-top:0}.q .t{font-weight:600}.q .h{color:var(--mute);font-size:14px;margin:2px 0 8px}
.opts{display:flex;gap:8px;flex-wrap:wrap}.opts label{border:1px solid var(--line);border-radius:8px;padding:6px 12px;cursor:pointer;user-select:none}
.opts input{position:absolute;opacity:0}.opts label:has(input:checked){background:var(--accbg);border-color:var(--acc);font-weight:600}
.opts label:has(input:focus-visible){outline:2px solid var(--acc)}
.sec{font-weight:700;margin:14px 0 2px}.miss{color:var(--warn);font-size:14px;margin-top:8px;min-height:1.3em}
textarea{width:100%;min-height:60px;font:inherit;background:var(--card);color:var(--ink);border:1px solid var(--line);border-radius:8px;padding:8px}
.pairs{display:grid;gap:12px;grid-template-columns:1fr}@media(min-width:700px){.pairs{grid-template-columns:1fr 1fr}}
.tag{display:inline-block;font-size:12px;padding:1px 8px;border-radius:99px;background:var(--accbg);color:var(--acc);margin-left:6px}
.intro{max-width:760px;margin:40px auto;padding:0 16px}.intro li{margin:6px 0}
</style></head><body>
<header><b id="ttl"></b><div class="bar"><i id="bar"></i></div><span id="cnt"></span>
<button id="prev">&larr; Back</button><button id="next" class="p">Save &amp; next &rarr;</button><button id="exp">Download my ratings (CSV)</button></header>
<div id="app"></div>
<script>
const DATA = __DATA__;
const A = DATA.annotator, MODE = DATA.mode, ITEMS = DATA.items, RUBRIC = DATA.rubric, REASONS = DATA.pair_reasons, COLS = DATA.columns;
const KEY = 'fjb_' + MODE + '_' + A;
let st = {i:0, ans:{}, secs:{}}, t0 = Date.now(), shown = false;
try { const s = localStorage.getItem(KEY); if (s) st = JSON.parse(s); } catch(e) {}
function save(){ try{ localStorage.setItem(KEY, JSON.stringify(st)); }catch(e){} }
const esc = s => String(s).replace(/[&<>"]/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
const done = id => st.ans[id] && st.ans[id]._ok;
function need(it){
  const a = (st.ans[it.case_id] ||= {}); const miss = [];
  if (it.item_type === 'pair'){ if(!a.pair_choice) miss.push('Which explanation is better?'); if(!a.pair_reason) miss.push('Main reason'); return miss; }
  RUBRIC.forEach(r => { if (!a[r.id]) miss.push(r.label); });
  return miss;
}
function render(){
  const app = document.getElementById('app');
  document.getElementById('ttl').textContent = (MODE==='pilot'?'PILOT - ':'') + 'Annotator ' + A;
  const n = ITEMS.length, d = ITEMS.filter(x => done(x.case_id)).length;
  document.getElementById('cnt').textContent = (st.i+1) + ' / ' + n + '  (' + d + ' done)';
  document.getElementById('bar').style.width = (100*d/n) + '%';
  document.getElementById('prev').disabled = st.i === 0;
  const it = ITEMS[st.i]; const a = (st.ans[it.case_id] ||= {});
  t0 = Date.now();
  const prof = '<div class="card"><h3>1. The person (profile)</h3><ul>' + it.profile_card.map(x => '<li>'+esc(x)+'</li>').join('') + '</ul></div>';
  const rec = '<div class="card"><h3>2. The recipe</h3><b>' + esc(it.recipe.title) + '</b><ul>' + it.recipe.ingredients.map(x => '<li>'+esc(x)+'</li>').join('') + '</ul></div>';
  const ev = '<div class="card"><h3>3. Checked facts about the recipe (trust these)</h3><ul>' + it.evidence.map(x => '<li>'+esc(x)+'</li>').join('') + '</ul></div>';
  let right = '';
  if (it.item_type === 'pair'){
    right = '<div class="card"><h3>4. Which explanation is better for this person?</h3><div class="pairs"><div class="card expl"><b>Explanation A</b><p>'+esc(it.explanation_A)+'</p></div><div class="card expl"><b>Explanation B</b><p>'+esc(it.explanation_B)+'</p></div></div>'
      + q('Which one is better?', 'Judge accuracy, honesty about downsides and fit with the profile - not which dish you prefer.', 'pair_choice', [['A','A is better'],['B','B is better'],['tie','About equal']], a)
      + q('Main reason', '', 'pair_reason', REASONS.map(r => [r,r]), a) + cm(a) + '<div class="miss" id="miss"></div></div>';
  } else {
    right = '<div class="card expl"><h3>4. The explanation to rate</h3><p>' + esc(it.explanation) + '</p></div><div class="card">';
    let sec = '';
    RUBRIC.forEach(r => {
      if (r.section !== sec){ sec = r.section; right += '<div class="sec">' + esc(sec) + '</div>'; }
      let o = [];
      if (r.type==='yn'||r.type==='yn_na') o = [['yes','Yes'],['no','No']];
      if (r.type==='yn_na') o.push(['na','Not applicable']);
      if (r.type==='scale'||r.type==='scale_na') o = [1,2,3,4,5].map(v => [String(v), String(v)]);
      if (r.type==='scale_na') o.push(['na','Not applicable']);
      right += q(r.label, r.q + (r.na ? '  [Not applicable = ' + r.na + ']' : ''), r.id, o, a);
    });
    right += cm(a) + '<div class="miss" id="miss"></div></div>';
  }
  app.innerHTML = '<main><div>' + prof + '<div style="height:16px"></div>' + rec + '<div style="height:16px"></div>' + ev + '</div><div>' + right + '</div></main>';
  app.querySelectorAll('input[type=radio]').forEach(el => el.addEventListener('change', e => { a[e.target.name] = e.target.value; save(); document.getElementById('miss').textContent=''; }));
  const c = app.querySelector('textarea'); if (c) c.addEventListener('input', e => { a.comment = e.target.value; save(); });
  window.scrollTo(0,0);
}
function q(label, hint, name, opts, a){
  return '<div class="q"><div class="t">'+esc(label)+'</div>'+(hint?'<div class="h">'+esc(hint)+'</div>':'')+'<div class="opts">'
    + opts.map(([v,t]) => '<label><input type="radio" name="'+name+'" value="'+esc(v)+'"'+(a[name]===v?' checked':'')+'>'+esc(t)+'</label>').join('') + '</div></div>';
}
function cm(a){ return '<div class="q"><div class="t">Comment (optional)</div><div class="h">Anything confusing, or why you chose a rating. Write which part of the text if you can.</div><textarea>'+esc(a.comment||'')+'</textarea></div>'; }
function commit(){
  const it = ITEMS[st.i]; const a = st.ans[it.case_id]; const m = need(it);
  st.secs[it.case_id] = (st.secs[it.case_id]||0) + Math.round((Date.now()-t0)/1000);
  if (m.length){ document.getElementById('miss').textContent = 'Please answer: ' + m.join(', ') + '.'; a._ok = false; save(); return false; }
  a._ok = true; save(); return true;
}
document.getElementById('next').onclick = () => { if (!commit()) return; if (st.i < ITEMS.length-1){ st.i++; save(); render(); } else { alert('All items done. Click "Download my ratings (CSV)" and send the file to the project lead.'); } };
document.getElementById('prev').onclick = () => { st.secs[ITEMS[st.i].case_id] = (st.secs[ITEMS[st.i].case_id]||0) + Math.round((Date.now()-t0)/1000); if (st.i>0){ st.i--; save(); render(); } };
document.getElementById('exp').onclick = () => {
  const rows = [COLS.join(',')]; const f = v => { v = (v===undefined||v===null)?'':String(v); return /[",\n]/.test(v) ? '"'+v.replace(/"/g,'""')+'"' : v; };
  ITEMS.forEach(it => { const a = st.ans[it.case_id]; if (!a || !a._ok) return;
    const r = {annotator_id:A, mode:MODE, case_id:it.case_id, item_type:it.item_type, comment:a.comment||'', seconds:st.secs[it.case_id]||'', saved_at:new Date().toISOString()};
    RUBRIC.forEach(x => r[x.id] = it.item_type==='pair' ? '' : (a[x.id]||'')); r.pair_choice = a.pair_choice||''; r.pair_reason = a.pair_reason||'';
    rows.push(COLS.map(c => f(r[c])).join(',')); });
  const b = new Blob(['﻿'+rows.join('\n')], {type:'text/csv'}); const u = URL.createObjectURL(b); const l = document.createElement('a');
  l.href = u; l.download = 'ratings_' + MODE + '_' + A + '.csv'; l.click(); URL.revokeObjectURL(u);
};
let firstUndone = ITEMS.findIndex(x => !done(x.case_id)); if (firstUndone > 0 && st.i === 0) st.i = firstUndone;
render();
</script></body></html>
"""


def form_items(ids):
    pub = {i["case_id"]: i for i in public_items}
    return [pub[c] for c in ids]


def write_form(annotator, mode, ids):
    payload = {"annotator": annotator, "mode": mode, "items": form_items(ids), "rubric": RUBRIC, "pair_reasons": PAIR_REASONS, "columns": template_cols}
    data = json.dumps(payload, ensure_ascii=False).replace("</", "<\\/")
    (FORMS / f"{mode}_{annotator}.html").write_text(FORM_HTML.replace("__DATA__", data), encoding="utf-8")


for a in ANNOTATORS:
    write_form(a, "pilot", pilot)
    write_form(a, "main", assign[a])
    pd.DataFrame({"case_id": assign[a]}).to_csv(OUT / f"assignment_{a}.csv", index=False)
print("forms written to", FORMS)
for p in sorted(FORMS.glob("*.html")):
    print(f"  {p.name}  {p.stat().st_size/1e6:.2f} MB")

forms written to C:\HP Data\University-Masters\Semester_3\P1_Natural Language Processing\Lab\Project\FoodJudgeBench\processed\case_bank\forms
  main_A1.html  1.04 MB
  main_A2.html  1.04 MB
  main_A3.html  1.04 MB
  pilot_A1.html  0.03 MB
  pilot_A2.html  0.03 MB
  pilot_A3.html  0.03 MB


#### 12. Final report
Check the numbers, then (1) read a few cases in a form yourself, (2) run the pilot (see `docs/08_annotator_guide.md`), (3) only then send the `main_*.html` forms.

In [14]:
rep = {
    "normal_cases": len(main_ids), "diagnostics": len(diag_ids), "pairs": len(pair_ids),
    "diagnostic_types": Counter(k["injected_error"] for k in keys if k["item_type"] == "diagnostic"),
    "items_per_annotator": {a: len(assign[a]) for a in ANNOTATORS},
    "rate_all": RATE_ALL, "pilot_items": len(pilot),
    "t3_equals_t4_cases": int(sum(1 for k in keys if k["technique"] == "grounded_check_llm" and k["t3_equals_t4"] is True)),
    "normal_cases_by_source": dict(Counter(k["profile_source"] for k in keys if k["item_type"] == "case") ),
    "seed": SEED,
}
rep["diagnostic_types"] = dict(rep["diagnostic_types"])
(OUT / "case_bank_report.json").write_text(json.dumps(rep, indent=1))
print(json.dumps(rep, indent=1))

{
 "normal_cases": 480,
 "diagnostics": 15,
 "pairs": 15,
 "diagnostic_types": {
  "invented_preference": 4,
  "wrong_nutrition": 4,
  "ignored_time_limit": 4,
  "ignored_dietary_constraint": 3
 },
 "items_per_annotator": {
  "A1": 510,
  "A2": 510,
  "A3": 510
 },
 "rate_all": true,
 "pilot_items": 10,
 "t3_equals_t4_cases": 83,
 "normal_cases_by_source": {
  "historical": 240,
  "synthetic": 240
 },
 "seed": 2026
}
